# Explainable Neural Trees for RUL Prediction in Turbofan Engines
## Under Sensor Noise and Missing Data Conditions

**Dataset:** NASA CMAPSS — Turbofan Engine Degradation Simulation  
**Paper submitted to:** IEEE SMC 2026 Workshop on Trustworthy AI for Safety-Critical Systems

---

### Experiments
1. **Baseline Comparison** — Neural Tree Ensemble vs Random Forest vs XGBoost
2. **Sensor Noise Robustness** — Performance under increasing Gaussian noise
3. **Missing Sensor Scenario** — Performance under random sensor dropout

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from src.data_preprocessing import load_cmapss, prepare_features, USEFUL_FEATURES
from src.experiments import run_experiment1, run_experiment2, run_experiment3
from src.visualization import (
    plot_rul_comparison, plot_metrics_bar, plot_noise_robustness,
    plot_missing_sensor_robustness, plot_feature_importance,
    plot_training_loss, plot_rul_scatter
)

print('All imports OK.')
print(f'Working directory: {os.getcwd()}')

## Step 1 — Load & Explore CMAPSS Dataset

We use **FD001** (single operating condition, single fault mode) as primary dataset.
- 100 training engines, 100 test engines
- 21 sensor channels + 3 operational settings
- RUL capped at 130 cycles (piece-wise linear target)

In [ ]:
DATA_DIR = '../data'
DATASET = 'FD001'

train_df, test_df = load_cmapss(dataset=DATASET, data_dir=DATA_DIR)

print(f'Training set: {train_df.shape}')
print(f'Test set:     {test_df.shape}')
print(f'\nTraining RUL distribution:')
print(train_df['RUL'].describe().round(2))
print(f'\nFeatures used: {USEFUL_FEATURES}')
print(f'Number of features: {len(USEFUL_FEATURES)}')

In [ ]:
# RUL distribution plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(train_df['RUL'], bins=30, color='#2196F3', alpha=0.8, edgecolor='black', linewidth=0.5)
axes[0].set_title('Training Set — RUL Distribution', fontweight='bold')
axes[0].set_xlabel('Remaining Useful Life (cycles)')
axes[0].set_ylabel('Frequency')
axes[0].grid(True, alpha=0.3)

# Sensor degradation for one engine
engine_1 = train_df[train_df['unit_id'] == 1].copy()
axes[1].plot(engine_1['time_cycle'], engine_1['s7'], color='#FF5722', linewidth=1.5, label='Sensor s7')
axes[1].plot(engine_1['time_cycle'], engine_1['s11'], color='#4CAF50', linewidth=1.5, label='Sensor s11')
axes[1].plot(engine_1['time_cycle'], engine_1['s12'], color='#9C27B0', linewidth=1.5, label='Sensor s12')
axes[1].set_title('Engine #1 — Sensor Degradation Over Time', fontweight='bold')
axes[1].set_xlabel('Flight Cycle')
axes[1].set_ylabel('Sensor Reading (normalized)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/figures/fig0_data_exploration.png', dpi=150, bbox_inches='tight')
plt.show()
print('Data exploration figure saved.')

In [ ]:
# Prepare feature matrices
X_train, y_train, X_test, y_test, scaler = prepare_features(train_df, test_df)
INPUT_DIM = X_train.shape[1]

print(f'X_train: {X_train.shape}  |  y_train: {y_train.shape}')
print(f'X_test:  {X_test.shape}   |  y_test:  {y_test.shape}')
print(f'Input dimension: {INPUT_DIM}')

---
## Experiment 1 — Baseline RUL Prediction Comparison

We compare three models:
- **Neural Tree Ensemble** (3× Soft Decision Trees, depth=5)
- **Random Forest** (200 trees)
- **XGBoost** (300 estimators, gradient boosting)

Metrics: RMSE, MAE, R²

In [ ]:
results = run_experiment1(
    X_train, y_train, X_test, y_test,
    input_dim=INPUT_DIM,
    epochs=150,
    verbose=True
)

# Summary table
summary = pd.DataFrame({
    name: {k: v for k, v in metrics.items() if k in ['RMSE','MAE','R2']}
    for name, metrics in results.items()
}).T
print('\n=== RESULTS SUMMARY ===')
print(summary.to_string())

In [ ]:
# Generate paper figures for Experiment 1
plot_rul_comparison(y_test, results, max_samples=100)
plot_metrics_bar(results)
plot_rul_scatter(y_test, results)
plot_training_loss(results['Neural Tree']['history'])
print('Experiment 1 figures saved.')

---
## Experiment 2 — Sensor Noise Robustness

We add **Gaussian noise** (σ = 0 to 0.20) to test sensor readings after training on clean data.
This simulates real-world sensor degradation, electromagnetic interference, and calibration drift.

All models are trained once on clean data and evaluated on increasingly noisy inputs.

In [ ]:
noise_df = run_experiment2(
    X_train, y_train, X_test, y_test,
    input_dim=INPUT_DIM,
    noise_levels=[0.0, 0.02, 0.05, 0.10, 0.15, 0.20],
    epochs=150,
    verbose=False
)

print('\n=== NOISE ROBUSTNESS RESULTS ===')
display_cols = ['noise_std', 'NT_RMSE', 'RF_RMSE', 'XGB_RMSE',
                'NT_R2', 'RF_R2', 'XGB_R2']
print(noise_df[display_cols].to_string(index=False))

In [ ]:
plot_noise_robustness(noise_df)
print('Experiment 2 figures saved.')

---
## Experiment 3 — Missing Sensor Scenario

We **zero-out random sensor channels** (0% to 50%) to simulate catastrophic sensor failure.
This is common in aircraft: a sensor may stop transmitting entirely during flight.

Models are trained on complete data and evaluated on incomplete sensor readings.

In [ ]:
missing_df = run_experiment3(
    X_train, y_train, X_test, y_test,
    input_dim=INPUT_DIM,
    missing_ratios=[0.0, 0.1, 0.2, 0.3, 0.4, 0.5],
    epochs=150,
    verbose=False
)

print('\n=== MISSING SENSOR RESULTS ===')
display_cols = ['missing_ratio', 'n_dropped', 'NT_RMSE', 'RF_RMSE', 'XGB_RMSE',
                'NT_R2', 'RF_R2', 'XGB_R2']
print(missing_df[display_cols].to_string(index=False))

In [ ]:
plot_missing_sensor_robustness(missing_df)
print('Experiment 3 figures saved.')

---
## Explainability — Feature Importance

Neural Trees provide **input-gradient-based feature importance**.
We identify which sensors drive RUL predictions — critical for safety-critical decisions.

In [ ]:
import torch

nt_model = results['Neural Tree']['model']
X_sample = X_test[:200]
importance = nt_model.get_feature_importance(torch.tensor(X_sample, dtype=torch.float32))

plot_feature_importance(importance, USEFUL_FEATURES, top_n=10)

# Print top 5 most important features
top_idx = np.argsort(importance)[::-1][:5]
print('Top 5 most important features:')
for rank, i in enumerate(top_idx, 1):
    print(f'  {rank}. {USEFUL_FEATURES[i]:8s} — importance: {importance[i]:.4f}')

---
## Summary Table — All Experiments

This table is used directly in the paper (Table I).

In [ ]:
print('='*65)
print('TABLE I — Experiment 1: Baseline Performance (CMAPSS FD001)')
print('='*65)
print(f'{"Model":<22} {"RMSE":>8} {"MAE":>8} {"R²":>8}')
print('-'*50)
for name in ['Neural Tree', 'Random Forest', 'XGBoost']:
    m = results[name]
    print(f'{name:<22} {m["RMSE"]:>8.3f} {m["MAE"]:>8.3f} {m["R2"]:>8.4f}')

print('\n' + '='*65)
print('TABLE II — Experiment 2: RMSE Under Sensor Noise')
print('='*65)
print(f'{"Noise σ":<10} {"Neural Tree":>14} {"Random Forest":>14} {"XGBoost":>10}')
print('-'*50)
for _, row in noise_df.iterrows():
    print(f'{row["noise_std"]:<10.2f} {row["NT_RMSE"]:>14.3f} {row["RF_RMSE"]:>14.3f} {row["XGB_RMSE"]:>10.3f}')

print('\n' + '='*65)
print('TABLE III — Experiment 3: RMSE Under Missing Sensors')
print('='*65)
print(f'{"Missing %":<12} {"Neural Tree":>14} {"Random Forest":>14} {"XGBoost":>10}')
print('-'*50)
for _, row in missing_df.iterrows():
    pct = f"{int(row['missing_ratio']*100)}%"
    print(f'{pct:<12} {row["NT_RMSE"]:>14.3f} {row["RF_RMSE"]:>14.3f} {row["XGB_RMSE"]:>10.3f}')

In [ ]:
# Save all results to CSV for paper writing
os.makedirs('../results', exist_ok=True)

# Baseline metrics
baseline_summary = pd.DataFrame({
    name: {k: v for k, v in metrics.items() if k in ['RMSE','MAE','R2']}
    for name, metrics in results.items()
}).T
baseline_summary.to_csv('../results/table1_baseline.csv')

# Noise robustness
noise_df.to_csv('../results/table2_noise_robustness.csv', index=False)

# Missing sensor
missing_df.to_csv('../results/table3_missing_sensor.csv', index=False)

print('All results saved to results/ directory.')
print('Figures saved to results/figures/ directory.')
print('\nExperiments complete!')